In [1]:
!pip -q install torch transformers datasets peft accelerate bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 34.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.9/518.9 kB 38.2 MB/s eta 0:00:00


In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import os

In [3]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

DATA_PATH = "/kaggle/input/week8-day2/train.jsonl"
VAL_PATH = "/kaggle/input/week8-day2/val.jsonl"
OUTPUT_DIR = "/kaggle/working/adapters"

os.makedirs(OUTPUT_DIR, exist_ok=True)
            
BATCH_SIZE = 4
EPOCHS = 3
LR = 2e-4
MAX_SEQ_LEN = 512

In [4]:
dataset = load_dataset(
    "json",
    data_files={"train": DATA_PATH, "validation": VAL_PATH},
)

dataset

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 1080
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 120
    })
})

In [5]:
def format_prompt(sample):
    return f"""### Instruction:
{sample['instruction']}

### Input:
{sample['input']}

### Response:
{sample['output']}"""

In [6]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [7]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

In [8]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map = "auto"
)

model.config.use_cache = False

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [9]:
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)

model.gradient_checkpointing_enable()

In [10]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj",]
    # "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

In [11]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


In [12]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=2,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    logging_steps=25,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    report_to="none",
    fp16=False,
    bf16=True,
    max_grad_norm=1.0,
    disable_tqdm=False,
)

In [13]:
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id
model.config.eos_token_id = tokenizer.eos_token_id

In [14]:
from transformers import DataCollatorForLanguageModeling

collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    formatting_func=format_prompt,
    data_collator=collator,
    args=training_args,
)

trainer.train()

Applying formatting function to train dataset:   0%|          | 0/1080 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/1080 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1080 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1080 [00:00<?, ? examples/s]

Applying formatting function to eval dataset:   0%|          | 0/120 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/120 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/120 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
25,1.814900
50,1.561800
75,1.476400
100,1.429500
125,1.420800
150,1.385700
175,1.369100
200,1.349700
225,1.324900
250,1.327200


TrainOutput(global_step=405, training_loss=1.3979650261961383, metrics={'train_runtime': 9374.1987, 'train_samples_per_second': 0.346, 'train_steps_per_second': 0.043, 'total_flos': 1.5426120408072192e+16, 'train_loss': 1.3979650261961383, 'entropy': 1.3870569467544556, 'num_tokens': 1508130.0, 'mean_token_accuracy': 0.6679639101028443, 'epoch': 3.0})

In [15]:
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

('/kaggle/working/adapters/tokenizer_config.json',
 '/kaggle/working/adapters/special_tokens_map.json',
 '/kaggle/working/adapters/chat_template.jinja',
 '/kaggle/working/adapters/tokenizer.model',
 '/kaggle/working/adapters/added_tokens.json',
 '/kaggle/working/adapters/tokenizer.json')

In [17]:
model.print_trainable_parameters()

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


In [19]:
from threading import Thread
from transformers import TextIteratorStreamer

model.eval()
model.config.use_cache = True

test_prompt = """### Instruction:
Analyze the medical scenario step by step, clearly explain the clinical reasoning, and conclude with the most likely diagnosis or finding.

### Input:
A 30-year-old female with a history of chronic cyclical abdominal pain that worsens during her menstrual cycle, and who has been married for 2 years without conceiving, presents to the clinic. What is the most appropriate next step in her management to investigate the cause of her symptoms?
### Response:
"""

inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)

streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

generation_kwargs = dict(
    **inputs,
    streamer=streamer,
    max_new_tokens=1200,  
    temperature=0.1,   
    repetition_penalty=1.2,
    do_sample=True,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id
)

thread = Thread(target=model.generate, kwargs=generation_kwargs)
thread.start()

print("### Response:")
for new_text in streamer:
    print(new_text, end="", flush=True)

thread.join()

### Response:
Okay, let's think about this. A woman with chronic abdominal pain during her menstrual cycle—that sounds like she might be dealing with something related to her period. So, what could it be? Let me start thinking through some possibilities here.

First off, I know there are several things that can trigger these kinds of cramps. For example, hormonal changes, stress, and even certain medications can make people feel really uncomfortable. But then there’s also the possibility of an underlying condition. Maybe she’s got something going on with her digestive system, maybe she’s just not getting enough nutrients from her diet.

Now, when we talk about digestion issues, they tend to have more than one root cause. There’s often a connection between gut health and how women experience their periods. It’s known that certain bacteria can play a role in regulating ovulation and fertility, so if those aren’t working right, you may see some kind of disruption in the timing of your cyc

In [20]:
import shutil
import os
shutil.make_archive('tiny_llama_finetuned', 'zip', '/kaggle/working/adapters')

print(f"Zip file created at: {os.getcwd()}/tiny_llama_finetuned.zip")

Zip file created at: /kaggle/working/tiny_llama_finetuned.zip
